[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/notebooks/profiling_and_tuning.ipynb)

# OpenImpala — Profiling and Performance Tuning

This notebook is the diagnostic complement to the seven user tutorials. Where the tutorials show *what* to do, this one shows *why* a given solve takes the time it does — and what to change when it takes too long.

Every `oi.tortuosity(numpy_array, ...)` call rebuilds the full AMReX + HYPRE stack from scratch and runs a percolation flood fill before the linear solve. That per-call setup cost can dominate small and medium problems. The first half of the notebook isolates and quantifies that cost; the second half goes below the binding layer with C++ and GPU profilers.

## Contents

| Section | Topic |
|---:|---|
| 1 | Setup — environment detection and dependency install |
| 1a | Build verification — confirm the GPU build is actually active |
| 2 | Synthetic datasets at 16³ – 128³ |
| 3 | Stage-by-stage timing of a single solve |
| 3b | Per-stage scaling: O(N) compute vs constant overhead |
| 4 | Fixed-overhead extraction via `t(N) = a + b·Nᵖ` |
| 5 | Amortization: reusing the VoxelImage across solves |
| 6 | cProfile — Python-level hot frames |
| 7 | AMReX TinyProfiler — C++ function breakdown |
| 8 | NVIDIA Nsight Systems — GPU kernel timeline |
| 9 | Solver comparison and scaling validation |
| 10 | `max_grid_size` tuning |
| 11 | Native-binary comparison (optional) |
| 12 | Summary and HPC recommendations |

Sections 1 – 6 run on any environment in a few minutes. Sections 7+ produce the most useful output on a Colab GPU runtime (T4 / A100 / L4).

## 1. Setup

Detect hardware, install dependencies, import, start a session.


In [ ]:
import subprocess
try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                       capture_output=True, text=True, timeout=10)
    gpu_name = r.stdout.strip()
    print(f"GPU detected: {gpu_name}" if gpu_name else "No GPU — CPU only.")
except FileNotFoundError:
    gpu_name = ""
    print("nvidia-smi not found — CPU only.")


In [ ]:
# OpenImpala ships two PyPI wheels:
#   openimpala       — pure-Python + SciPy fallback (CPU only)
#   openimpala-cuda  — compiled C++ HYPRE/AMReX backend with CUDA kernels
#
# On a GPU runtime we install the CUDA wheel; otherwise we fall back to the
# pure-Python package.
!apt-get install -y libopenmpi-dev > /dev/null 2>&1
if gpu_name:
    print("GPU detected — installing openimpala-cuda")
    !pip install -q openimpala-cuda porespy py-spy
else:
    print("No GPU detected — installing openimpala (CPU)")
    !pip install -q openimpala porespy py-spy
print("Dependencies installed.")

In [ ]:
import openimpala as oi
from openimpala import core as oic  # noqa: F401
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, io, sys, re, cProfile, pstats, warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print(f"OpenImpala version: {oi.__version__}")


In [ ]:
session = oi.Session()
session.__enter__()
print("OpenImpala session started.")


## 1a. Build verification

Before profiling anything, confirm that the wheel `pip` installed actually has the GPU backend you expect. The most common cause of "OpenImpala feels slow on Colab" is a CPU-only wheel running on a GPU-equipped host — every solve in the rest of the notebook would be CPU-bound and ~10–50× slower than necessary on a single-phase 3D diffusion problem at 128³+.

`oi.build_info()` returns a dict of compile-time feature flags plus the runtime GPU device count. We check that the backend matches the hardware, and warn loudly if it doesn't.

In [ ]:
info = oi.build_info()
backend = info["backend"]   # one of: cpp-cuda, cpp-hip, cpp-cpu, pure-python

print(f"openimpala backend: {backend}")
print(f"  version:        {info.get('version', 'unknown')}")
print(f"  OpenMP threads: {info.get('openmp_max_threads', 1)}")
print(f"  MPI enabled:    {info.get('mpi_enabled', False)}")
print(f"  HYPRE CUDA:     {info.get('hypre_cuda', False)}")
print(f"  TinyProfile:    {info.get('tiny_profile', False)}")
if info.get("gpu_device_count", -1) > 0:
    print(f"  GPU devices:    {info['gpu_device_count']}")

is_gpu_build = backend in ("cpp-cuda", "cpp-hip")

if gpu_name and not is_gpu_build:
    print()
    print("=" * 70)
    print("  WARNING: CPU-only build detected on a GPU-equipped host")
    print("=" * 70)
    print(f"  Host GPU:          {gpu_name}")
    print(f"  openimpala build:  {backend}")
    print()
    print("  The GPU is idle. Every solve in this notebook will run on the CPU,")
    print("  which is typically 10–50× slower than the CUDA build for")
    print("  single-phase 3D diffusion on 128³+ grids.")
    print()
    print("  To fix:")
    print("    !pip uninstall -y openimpala")
    print("    !pip install openimpala-cuda")
    print("  Then Runtime → Restart session and re-run from the top.")
    print("=" * 70)
elif is_gpu_build:
    print(f"\n{backend.upper()} build active on {gpu_name or 'accelerator'}.")
else:
    print("\nCPU-only run (no accelerator detected).")

## 2. Synthetic datasets

PoreSpy is used to generate five synthetic porous structures at 16³, 32³, 64³, 96³, and 128³. Sizes are kept small enough that a full profiling run completes in minutes rather than hours, and large enough that the cubic scaling of voxel count is visible in the timings.

The 16³ dataset is small enough that its solve time is dominated by per-call setup rather than compute — useful for extracting the fixed overhead in §4.

In [ ]:
import porespy as ps

def make_micro(size, porosity=0.5, seed=42):
    np.random.seed(seed)
    im = ps.generators.blobs(shape=[size] * 3, porosity=porosity, blobiness=1.5)
    return im.astype(np.int32)

sizes = [16, 32, 64, 96, 128]
datasets = {n: make_micro(n) for n in sizes}
data_medium = datasets[64]  # default workhorse for most diagnostics

for n, d in datasets.items():
    print(f"  {n:>4d}^3  porosity={np.mean(d == 0):.3f}  bytes={d.nbytes/1e6:.2f} MB")


## 3. Python-level stage breakdown — one call, one bar chart

`oi.tortuosity(numpy_array, ...)` is not a single operation. From
`facade.py` it is roughly:

```
numpy → VoxelImage.from_numpy    (copy + AMReX BoxArray/Geometry/iMultiFab build)
PercolationCheck(img, ...)        (flood fill — runs every call!)
VolumeFraction(img, ...)          (cheap count)
TortuosityHypre(img, ..., solver) (HYPRE grid/stencil/matrix assembly)
solver.value()                    (actual linear solve + flux integration)
```

We time each stage independently. Whichever bar dominates *is* the
bottleneck. If `from_numpy` or `TortuosityHypre(...)` construction dominates,
your slowdown is in binding/setup — not the solver.

**Note on solver choice for this section**: we time PCG+SMG below because
it's the most direct apples-to-apples comparison with a CPU solver and
its iteration count is informative. On GPU hardware **MLMG is usually
2–5× faster** for this problem class (matrix-free, no assembly cost,
fully GPU-native kernels). §9 runs the full bake-off including MLMG so
you can see the numbers side-by-side.

In [ ]:
from openimpala import _core as _core
from openimpala.facade import _parse_direction, _parse_solver

def time_stages(data, solver="pcg", direction="z", max_grid_size=32, maxiter=1000):
    """Decompose a single tortuosity call into timed binding stages.

    Returns a dict of stage timings plus metadata. If the solver fails to
    converge or produces an invalid result (e.g. boundary flux conservation
    check tripped at large N), the dict still contains every successful stage
    timing and sets _tau / _iters to NaN — callers can then plot or skip the
    failed point without crashing.
    """
    d = _parse_direction(direction)
    st = _parse_solver(solver)
    out = {}

    t0 = time.perf_counter()
    arr = np.ascontiguousarray(data, dtype=np.int32)
    out["np.ascontiguousarray copy"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    img = _core.VoxelImage.from_numpy(arr, max_grid_size)
    out["VoxelImage.from_numpy (AMReX grid + voxel write)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    pc = _core.PercolationCheck(img, 0, d, 0)
    out["PercolationCheck (flood fill)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    vf_val = _core.VolumeFraction(img, 0, 0).value_vf()
    out["VolumeFraction"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    solver_obj = _core.TortuosityHypre(img, vf_val, 0, d, st, ".", 0.0, 1.0, 0, False)
    out["TortuosityHypre ctor (HYPRE setup + matrix assembly)"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    try:
        tau = solver_obj.value()
        out["solver.value() (solve + flux)"] = time.perf_counter() - t0
        out["_tau"] = tau
        out["_iters"] = solver_obj.iterations
    except RuntimeError as e:
        out["solver.value() (solve + flux)"] = time.perf_counter() - t0
        out["_tau"] = float("nan")
        out["_iters"] = solver_obj.iterations
        out["_error"] = str(e)

    out["_total"] = sum(v for k, v in out.items() if not k.startswith("_"))
    return out

# Warm-up — first call pays lazy-init inside AMReX/HYPRE
_ = time_stages(datasets[32])

stages = time_stages(data_medium, solver="pcg")
if np.isnan(stages["_tau"]):
    print(f"⚠ solve did not produce a valid tau ({stages.get('_error', 'unknown')}); "
          f"stage timings still shown.\n")
else:
    print(f"tau = {stages['_tau']:.4f}   iters = {stages['_iters']}   total = {stages['_total']:.3f}s\n")
for k, v in stages.items():
    if k.startswith("_"):
        continue
    print(f"  {k:55s}  {v:7.3f}s  ({100*v/stages['_total']:5.1f}%)")

In [ ]:
stage_names = [k for k in stages if not k.startswith("_")]
stage_times = [stages[k] for k in stage_names]

fig, ax = plt.subplots(figsize=(11, 4))
y = np.arange(len(stage_names))
colors = ["#9b59b6", "#8e44ad", "#2ecc71", "#95a5a6", "#f39c12", "#e74c3c"]
ax.barh(y, stage_times, color=colors[:len(stage_names)], alpha=0.85, edgecolor="white")
for i, t in enumerate(stage_times):
    ax.text(t + max(stage_times) * 0.01, i,
            f"{t:.3f}s  ({100*t/stages['_total']:.1f}%)", va="center", fontsize=9)
ax.set_yticks(y)
ax.set_yticklabels(stage_names, fontsize=9)
ax.set_xlabel("Wall time (s)")
ax.set_title("Single oi.tortuosity(64³) call — stage breakdown", fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

slowest = max(stage_names, key=lambda k: stages[k])
print(f"\nBottleneck at 64³: {slowest} ({100*stages[slowest]/stages['_total']:.1f}%)")


### §3b Per-stage scaling — does this stage scale with N, or is it fixed overhead?

Run the same decomposition at 32³ and 128³ and plot the ratio per stage.
A stage with ratio ≈ `(128/32)³ = 64×` is doing O(N) work — legitimate
compute. A stage with ratio ≈ 1× is constant overhead — every call pays
the same price regardless of problem size, which is the *classic* signature
of binding / setup cost.


In [ ]:
stages_small = time_stages(datasets[32])
stages_big   = time_stages(datasets[128])

if np.isnan(stages_big.get("_tau", 1.0)):
    print(f"Note: 128³ solve did not return a valid tau "
          f"({stages_big.get('_error', 'unknown')}); stage timings still usable "
          f"for the scaling-ratio analysis below.\n")

# Ideal O(N) ratio when scaling voxel count by (128/32)^3
N_ratio = (128 / 32) ** 3
print(f"Ideal O(N) work ratio for 32->128: {N_ratio:.0f}×")
print(f"{'stage':<55s} {'32³ (s)':>10s} {'128³ (s)':>10s} {'ratio':>8s}  behaviour")
print("-" * 110)

rows = []
for k in stage_names:
    ts, tb = stages_small[k], stages_big[k]
    r = tb / max(ts, 1e-9)
    if r > 0.5 * N_ratio:
        label = "scales O(N) ✓"
    elif r < 2.0:
        label = "~constant (overhead)"
    else:
        label = "sub-linear"
    rows.append({"stage": k, "t_32": ts, "t_128": tb, "ratio": r, "label": label})
    print(f"{k:<55s} {ts:>10.3f} {tb:>10.3f} {r:>8.1f}×  {label}")

df_scale = pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
y = np.arange(len(stage_names))
w = 0.4
ax.barh(y - w/2, df_scale["t_32"], w, color="#3498db", alpha=0.85, label="32³")
ax.barh(y + w/2, df_scale["t_128"], w, color="#e74c3c", alpha=0.85, label="128³")
for i, r in enumerate(df_scale["ratio"]):
    ax.text(df_scale["t_128"].iloc[i] * 1.02, i + w/2, f"{r:.0f}×",
            va="center", fontsize=8, color="#2c3e50")
ax.set_yticks(y)
ax.set_yticklabels(stage_names, fontsize=9)
ax.set_xlabel("Wall time (s)")
ax.set_title(f"Per-stage scaling (ideal O(N) ratio = {N_ratio:.0f}×)", fontweight="bold")
ax.axvline(0, color="#bdc3c7", linewidth=0.5)
ax.invert_yaxis()
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

constant_stages = df_scale[df_scale["ratio"] < 2.0]
if not constant_stages.empty:
    print("\n⚠ Stages that DON'T scale with problem size (pure overhead):")
    for _, r in constant_stages.iterrows():
        print(f"    {r['stage']:55s}  ratio={r['ratio']:.1f}×")
    print("  Every oi.tortuosity() call pays these, regardless of image size.")


## 4. Fixed-overhead extraction

A call's wall time has two parts: a fixed per-call overhead `a` (binding
dispatch, AMReX/HYPRE setup, flood fill) and a size-dependent compute part
`b·Nᵖ`. Fit `t(N) = a + b·Nᵖ` across sizes.

If `a` is a meaningful fraction of the medium-size solve time, every solve
you run is paying a tax that has nothing to do with the actual physics.


In [ ]:
# Sweep grid sizes, take median of two trials. The TortuosityHypre flux-
# conservation guard in oi.tortuosity occasionally rejects a HYPRE solve at
# larger N — GPU reductions are non-deterministic and the final residual can
# land just outside the flux-balance tolerance from one run to the next.
# Catch the resulting ConvergenceError and record NaN; the fit downstream
# drops NaN rows automatically.
from openimpala import ConvergenceError

overhead_rows = []
for n, d in datasets.items():
    ts = []
    fails = 0
    for _ in range(2):
        t0 = time.perf_counter()
        try:
            oi.tortuosity(d, phase=0, direction="z", solver="pcg",
                          max_grid_size=32, verbose=0)
            ts.append(time.perf_counter() - t0)
        except ConvergenceError:
            ts.append(time.perf_counter() - t0)
            fails += 1
    t_med = float(np.median(ts)) if ts else float("nan")
    note = "  (some trials failed)" if fails else ""
    overhead_rows.append({"n": n, "voxels": n**3, "t": t_med, "fails": fails})
    print(f"  {n:>4d}^3  {t_med:.3f}s{note}")

df_oh = pd.DataFrame(overhead_rows)
df_oh

In [ ]:
from scipy.optimize import curve_fit

# Fit only rows that produced a real timing (drop NaN where the guard
# tripped on every trial).
_mask = ~np.isnan(df_oh["t"].values)
N = df_oh["voxels"].values.astype(float)[_mask]
T = df_oh["t"].values[_mask]
if len(N) < 2:
    raise RuntimeError("Fewer than 2 successful timings — can't fit overhead model.")

def model(x, a, b, p):
    return a + b * np.power(x, p)

try:
    popt, _ = curve_fit(model, N, T, p0=[T.min(), 1e-9, 1.0], maxfev=20000)
    a_raw, b_fit, p_fit = popt
except Exception as e:
    print(f"Fit failed: {e}")
    a_raw, b_fit, p_fit = T.min(), 0.0, 1.0

# Clamp: a negative intercept means there is no meaningful fixed overhead —
# pretending otherwise misleads the summary.
a_fit = max(a_raw, 0.0)
overhead_meaningful = a_fit > 0.01 * T.max()

df_oh["overhead_pct"] = 100 * a_fit / df_oh["t"]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(N, T, "o", color="#2c3e50", markersize=10, label="Measured")
xs = np.logspace(np.log10(N.min()), np.log10(N.max()), 200)
ax.loglog(xs, model(xs, a_fit, b_fit, p_fit), "-", color="#3498db",
          label=f"Fit: {a_fit:.2f}s + {b_fit:.2e}·N^{p_fit:.2f}")
if overhead_meaningful:
    ax.axhline(a_fit, color="#e74c3c", linestyle="--",
               label=f"Fixed overhead ≈ {a_fit:.2f}s")
ax.set_xlabel("Voxels (N)")
ax.set_ylabel("Wall time (s)")
ax.set_title("Fixed overhead vs compute", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

if a_raw < 0:
    print(f"\nFit intercept was negative ({a_raw:.2f}s) — clamped to 0.")
    print("Interpretation: no detectable per-call fixed overhead; all time is compute.")
print(f"Estimated fixed overhead per call: {a_fit:.2f}s")
print(f"Compute scaling:                  O(N^{p_fit:.2f})")

if p_fit > 1.2:
    print(f"\n⚠ Super-linear compute scaling (O(N^{p_fit:.2f}) vs ideal O(N)).")
    print("  Likely cause: Krylov iteration count growing with N. A multigrid")
    print("  preconditioner (SMG/PFMG) or MLMG would bring this closer to O(N).")

print("\nOverhead share per size:")
print(df_oh[["n", "t", "overhead_pct"]].to_string(index=False))


## 5. Amortization test — reusing the VoxelImage

If most per-call time goes into rebuilding AMReX state, then running the solver multiple times against the *same* `VoxelImage` should be markedly cheaper than the equivalent number of calls through the high-level facade. This cell measures the gap.

Two paths are compared on `data_medium`, five calls each:

* **Naive**: five calls to `oi.tortuosity(numpy_array, ...)`. Every call rebuilds the geometry, BoxArray, distribution mapping, iMultiFab, percolation mask, and HYPRE matrix from the bare NumPy input.
* **Amortized**: one call to `_numpy_to_voxelimage` to build the `VoxelImage` once, then five direct calls to `_core.TortuosityHypre` reusing it.

The difference is the per-call setup tax. Sweeping (e.g. directional tortuosity, parameter studies) should always use the amortized pattern.

In [ ]:
from openimpala.facade import _numpy_to_voxelimage, _parse_direction, _parse_solver

def naive_repeat(data, n=5, solver="pcg"):
    """Top-level facade — rebuilds everything every call."""
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        oi.tortuosity(data, phase=0, direction="z", solver=solver,
                      max_grid_size=32, verbose=0)
        ts.append(time.perf_counter() - t0)
    return ts

def amortized_repeat(data, n=5, solver="pcg"):
    """Build VoxelImage once, reuse across solves."""
    d = _parse_direction("z")
    st = _parse_solver(solver)
    img = _numpy_to_voxelimage(data, 32)
    vf = _core.VolumeFraction(img, 0, 0).value_vf()

    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        s = _core.TortuosityHypre(img, vf, 0, d, st, ".", 0.0, 1.0, 0, False)
        s.value()
        ts.append(time.perf_counter() - t0)
    return ts

n_repeat = 5
naive = naive_repeat(data_medium, n_repeat)
amort = amortized_repeat(data_medium, n_repeat)

print(f"Naive  oi.tortuosity(numpy) × {n_repeat}: " + ", ".join(f"{t:.2f}s" for t in naive))
print(f"Amortized (reuse VoxelImage): " + ", ".join(f"{t:.2f}s" for t in amort))
print(f"\nNaive total:      {sum(naive):.2f}s  (mean {np.mean(naive):.2f}s/call)")
print(f"Amortized total:  {sum(amort):.2f}s  (mean {np.mean(amort):.2f}s/call)")
print(f"Speedup from reuse: {sum(naive)/max(sum(amort), 1e-9):.2f}×")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(1, n_repeat + 1)
ax.plot(x, naive, "o-", color="#e74c3c", linewidth=2, markersize=9,
        label="Naive: oi.tortuosity(numpy, ...)")
ax.plot(x, amort, "o-", color="#2ecc71", linewidth=2, markersize=9,
        label="Amortized: reuse VoxelImage + TortuosityHypre")
ax.set_xlabel("Call #")
ax.set_ylabel("Wall time (s)")
ax.set_title("Per-call cost: rebuild-everything vs reuse", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

gap = np.mean(naive) - np.mean(amort)
print(f"Average per-call rebuild tax: {gap:.2f}s "
      f"({100*gap/np.mean(naive):.0f}% of a naive call)")


## 6. cProfile — Python-level hot frames

cProfile captures cumulative time per Python frame, including time spent inside pybind11 wrappers around C++ calls. This is a different view from §3 — instead of timing named stages, it counts every function call and surfaces unexpected frame costs.

Things to look for in the top 20 frames:

* `_numpy_to_voxelimage`, `VoxelImage.from_numpy` — data ingestion
* `PercolationCheck.__init__` — flood-fill setup
* `TortuosityHypre.__init__` — HYPRE matrix assembly
* `TortuosityHypre.value` — the linear solve

If pure-Python helpers like `_parse_direction` or `_parse_solver` show up high, the binding is doing too much Python-side dispatch.

In [ ]:
pr = cProfile.Profile()
pr.enable()
oi.tortuosity(data_medium, phase=0, direction="z", solver="pcg",
              max_grid_size=32, verbose=0)
pr.disable()

stream = io.StringIO()
ps_stats = pstats.Stats(pr, stream=stream).sort_stats("cumulative")
ps_stats.print_stats(20)
print(stream.getvalue())


## 7. AMReX TinyProfiler — C++ function breakdown

§3 showed that `solver.value()` accounts for the bulk of wall time at 64³ and that it scales super-linearly to 128³. To break that down further we need to go below the binding layer, into AMReX's own profiler.

`AMReX::TinyProfiler` instruments every `BL_PROFILE`-annotated function in the C++ source and prints a tabular breakdown when AMReX finalises. The compiled wheel sets `-DAMReX_TINY_PROFILE=ON`, so the instrumentation is always on.

The profiler writes to C++ streams on `amrex::Finalize()`, which Python's `sys.stdout` redirection can't capture from inside a long-running session. We therefore run a single profiled solve in a short-lived subprocess — Python's `capture_output=True` collects stdout and stderr at the OS level and gives us the complete table.

In [ ]:
import subprocess, tempfile, os, textwrap, json as _json

def run_with_profiler(data, solver="pcg", direction="z", max_grid_size=32, timeout=600):
    """Run ONE profiled solve in a subprocess.

    Why a subprocess: TinyProfiler prints via C++ streams on AMReX finalize,
    which sys.stdout redirection can't capture. A subprocess gives us a clean
    AMReX lifecycle and Python's ``capture_output=True`` grabs fd 1 + fd 2
    at the OS level — including everything TinyProfiler emits.

    Returns (result_dict, combined_stdout_stderr).
    """
    tmp = tempfile.mkdtemp()
    npy_path = os.path.join(tmp, "data.npy")
    np.save(npy_path, data)

    script = textwrap.dedent(f"""
        import json, numpy as np, openimpala as oi
        data = np.load({npy_path!r})
        with oi.Session():
            r = oi.tortuosity(data, phase=0, direction={direction!r},
                              solver={solver!r}, max_grid_size={max_grid_size},
                              verbose=1)
            print("__RESULT__" + json.dumps({{
                "tau": float(r.tortuosity),
                "converged": bool(r.solver_converged),
                "iters": int(r.iterations),
                "residual": float(r.residual_norm),
            }}))
    """).strip()
    script_path = os.path.join(tmp, "run.py")
    with open(script_path, "w") as f:
        f.write(script)

    proc = subprocess.run([sys.executable, script_path],
                          capture_output=True, text=True, timeout=timeout)
    combined = (proc.stdout or "") + "\n" + (proc.stderr or "")

    # Pull the result line out of stdout
    result = None
    for line in combined.splitlines():
        if line.startswith("__RESULT__"):
            try:
                result = _json.loads(line[len("__RESULT__"):])
            except Exception:
                pass
            break

    if proc.returncode != 0:
        print(f"⚠ subprocess exited with code {proc.returncode}")
    return result, combined

print("Running a profiled solve on 32^3 in a subprocess…")
prof_result, prof_output = run_with_profiler(datasets[32], solver="pcg")
if prof_result is not None:
    print(f"tau={prof_result['tau']:.4f}  converged={prof_result['converged']}  iters={prof_result['iters']}")
else:
    print("No result line found — solve may have failed; check output below.")
print(f"Captured output length: {len(prof_output)} chars")

# Show the tail (TinyProfiler dumps near the end, on finalize)
if prof_output.strip():
    print("\n--- Last 4000 chars of captured output ---")
    print(prof_output[-4000:])
else:
    print("\n(No output captured.)")


In [ ]:
def parse_tiny_profiler(output):
    """Parse AMReX TinyProfiler output into a DataFrame."""
    records = []
    pattern = re.compile(
        r"^\s*(\S+.*\S)\s+"
        r"(\d+)\s+"
        r"([\d.e+-]+)\s+"
        r"([\d.e+-]+)\s+"
        r"([\d.e+-]+)\s+"
        r"([\d.e+-]+)\s*%",
        re.MULTILINE,
    )
    for m in pattern.finditer(output):
        records.append({
            "function": m.group(1).strip(),
            "ncalls": int(m.group(2)),
            "excl_min_s": float(m.group(3)),
            "excl_avg_s": float(m.group(4)),
            "excl_max_s": float(m.group(5)),
            "max_pct": float(m.group(6)),
        })
    return pd.DataFrame(records)

df_prof = parse_tiny_profiler(prof_output)

# Heuristic: TinyProfiler tables always contain "TinyProfiler" or
# "Name   NCalls" header. If neither is present, the build just doesn't
# emit a table and there is nothing to parse.
has_tp_header = ("TinyProfiler" in prof_output
                 or re.search(r"Name\s+NCalls", prof_output) is not None)

if len(df_prof) > 0:
    df_prof = df_prof.sort_values("excl_avg_s", ascending=False).head(15)
    print(f"Top {len(df_prof)} C++ functions by exclusive time:")
    print(df_prof.to_string(index=False))
elif not has_tp_header:
    print("=" * 70)
    print("  ⚠  No TinyProfiler table was emitted by this openimpala build")
    print("=" * 70)
    print("  The subprocess ran to completion (result captured above) but")
    print("  AMReX did not dump a TinyProfiler table on finalize. This means")
    print("  the C++ side was built without the profiling flag.")
    print()
    print("  To enable, rebuild openimpala's AMReX dependency with one of:")
    print("    CMake:  -DAMReX_TINY_PROFILE=ON")
    print("    GNUmake: TINY_PROFILE=TRUE")
    print()
    print("  Until then §7 cannot break down the C++ solve into components.")
    print("  §3 (Python stage breakdown) + §3b (per-stage scaling) already")
    print("  show that solver.value() is ~85-95% of wall time and scales")
    print("  super-linearly — that IS the bottleneck regardless.")
    print("=" * 70)
else:
    print("TinyProfiler header detected but no rows parsed — regex mismatch.")
    print("Dump tail of raw output above and adjust parse_tiny_profiler().")


In [ ]:
if len(df_prof) > 0 and "max_pct" in df_prof.columns:
    def categorize(name):
        n = name.lower()
        if "solve" in n:
            return "#e74c3c"
        if any(k in n for k in ["setup", "fill", "matrix", "stencil", "assemble"]):
            return "#f39c12"
        if any(k in n for k in ["flux", "plane"]):
            return "#3498db"
        if any(k in n for k in ["mask", "precondition", "flood", "percolation"]):
            return "#2ecc71"
        return "#95a5a6"

    colors = [categorize(f) for f in df_prof["function"]]
    y = np.arange(len(df_prof))

    fig, ax = plt.subplots(figsize=(10, max(4, len(df_prof) * 0.4)))
    ax.barh(y, df_prof["excl_avg_s"], color=colors, alpha=0.85, edgecolor="white")
    for i, (_, row) in enumerate(df_prof.iterrows()):
        ax.text(row["excl_avg_s"] + 0.001, i, f'{row["max_pct"]:.1f}%',
                va="center", fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(df_prof["function"], fontsize=9)
    ax.set_xlabel("Exclusive time (s)")
    ax.set_title("AMReX TinyProfiler — C++ function breakdown", fontweight="bold")
    ax.invert_yaxis()

    from matplotlib.patches import Patch
    legend = [
        Patch(facecolor="#e74c3c", label="Linear solve"),
        Patch(facecolor="#f39c12", label="Matrix assembly/setup"),
        Patch(facecolor="#3498db", label="Flux computation"),
        Patch(facecolor="#2ecc71", label="Pre-processing"),
        Patch(facecolor="#95a5a6", label="Other"),
    ]
    ax.legend(handles=legend, loc="lower right", framealpha=0.9)
    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart — no TinyProfiler data.")


## 8. NVIDIA Nsight Systems — GPU kernel timeline

If a GPU is available, `nsys` captures a kernel-level timeline: launches,
memory transfers, idle gaps. Signatures of Python-binding overhead on the
GPU path are:

- Many tiny launches with gaps → Python loop driving the GPU
- Frequent H2D transfers of the same data → `numpy → VoxelImage` copies leaking out
- Long idle gaps between kernels → CPU-side serial work between solver calls

We use a standalone script so the subprocess is clean.


In [ ]:
import shutil

# Try to find nsys. On Colab it isn't pre-installed but can be apt-get'd
# in a few seconds. Pull it now if missing so §8 doesn't silently skip.
if shutil.which("nsys") is None and bool(gpu_name):
    print("nsys not on PATH — apt-get installing nsight-systems...")
    !apt-get install -y nsight-systems-cli > /dev/null 2>&1 || \
        apt-get install -y nsight-systems > /dev/null 2>&1
    if shutil.which("nsys") is None:
        # Fall back to whichever nsys ships under /opt/nvidia or similar
        import glob
        for p in glob.glob("/opt/nvidia/nsight-systems*/bin/nsys") + \
                glob.glob("/usr/local/cuda*/bin/nsys"):
            import os
            os.environ["PATH"] = os.path.dirname(p) + ":" + os.environ["PATH"]
            break

nsys_available = shutil.which("nsys") is not None and bool(gpu_name)

if not nsys_available:
    if not gpu_name:
        print("No GPU — skipping Nsight. (Runtime > Change runtime type > T4 GPU)")
    else:
        print("nsys still not available after install attempt. "
              "Try: !apt-get install -y nsight-systems-cli")
else:
    print(f"nsys: {shutil.which('nsys')}   GPU: {gpu_name}")
    script = '''
import openimpala as oi, numpy as np, porespy as ps
np.random.seed(42)
data = ps.generators.blobs(shape=[64]*3, porosity=0.5, blobiness=1.5).astype(np.int32)
with oi.Session():
    r = oi.tortuosity(data, phase=0, direction="z", solver="pcg",
                      max_grid_size=32, verbose=0)
    print(f"tau={r.tortuosity:.4f} converged={r.solver_converged}")
'''
    with open("/tmp/oi_profile.py", "w") as f:
        f.write(script)
    !nsys profile --output=/tmp/oi_profile --force-overwrite=true \
        --trace=cuda,nvtx,osrt --cuda-memory-usage=true --stats=true \
        python3 /tmp/oi_profile.py 2>&1 | tail -60
    print("\nReport saved to /tmp/oi_profile.nsys-rep (download for Nsight GUI).")

In [ ]:
if nsys_available:
    stats = subprocess.run(
        ["nsys", "stats", "--report", "cuda_gpu_kern_sum",
         "--format", "csv", "/tmp/oi_profile.nsys-rep"],
        capture_output=True, text=True, timeout=60,
    )
    if stats.returncode == 0 and stats.stdout.strip():
        lines = stats.stdout.strip().split("\n")
        hdr = next((i for i, l in enumerate(lines) if "Time" in l and "Name" in l), None)
        if hdr is not None:
            df_k = pd.read_csv(io.StringIO("\n".join(lines[hdr:])))
            if "Total Time (ns)" in df_k.columns:
                df_k["Total Time (ms)"] = df_k["Total Time (ns)"] / 1e6
                df_k = df_k.sort_values("Total Time (ns)", ascending=False).head(10)
                print("Top CUDA kernels by total GPU time:")
                print(df_k[["Name", "Total Time (ms)", "Instances"]].to_string(index=False))
            else:
                print(df_k.head().to_string(index=False))
        else:
            print(stats.stdout[:2000])
    else:
        print("nsys stats produced no kernel data.")


In [ ]:
if nsys_available:
    mem = subprocess.run(
        ["nsys", "stats", "--report", "cuda_gpu_mem_size_sum",
         "--format", "csv", "/tmp/oi_profile.nsys-rep"],
        capture_output=True, text=True, timeout=60,
    )
    if mem.returncode == 0 and mem.stdout.strip():
        lines = mem.stdout.strip().split("\n")
        hdr = next((i for i, l in enumerate(lines)
                    if "Operation" in l or "Total" in l), None)
        if hdr is not None:
            df_mem = pd.read_csv(io.StringIO("\n".join(lines[hdr:])))
            print("CUDA memory transfer summary (H2D/D2H):")
            print(df_mem.to_string(index=False))
            print("\nRed flags:")
            print("  • Frequent H2D transfers of similar size → numpy→VoxelImage")
            print("    copy leaking out of bindings into the hot path")
            print("  • Large D2H transfer per call → result readback overhead")
        else:
            print(mem.stdout[:1500])
    else:
        print("nsys mem report produced no data.")


## 9. Solver comparison

With the bottlenecks identified, this section compares the available solver / preconditioner combinations on `data_medium` at a single grid size. Each solver runs once — the goal is to pick a baseline configuration, not to measure steady-state noise.

The combinations cover three classes of solver:

* **HYPRE Krylov methods** (`pcg`, `gmres`, `flexgmres`, `bicgstab`) with optional multigrid preconditioning. PCG is for symmetric problems (the natural choice for tortuosity); the others handle non-symmetric operators.
* **HYPRE standalone multigrid** (`pfmg`, `smg`) — direct multigrid solvers, no Krylov outer loop.
* **AMReX MLMG** (`mlmg`) — matrix-free geometric multigrid implemented natively in AMReX. Bypasses HYPRE entirely, so there is no matrix-assembly cost (which §3 showed accounts for ~20% of total time) and the entire solve runs on GPU device kernels. Typically the fastest option on CUDA hardware for structured-grid problems like tortuosity.

What to look for in the bar chart below: the best solver on a CPU build is usually `pcg+pfmg`; on a GPU build MLMG normally wins by 2–10×, with PFMG-preconditioned Krylov methods close behind. SMG-based combinations work on both CPU and GPU but are typically slower than PFMG.

In [ ]:
# Compare Krylov and multigrid choices on 64^3.
# Each entry is (label, solver, preconditioner). A preconditioner of None means
# the solver ignores it (standalone SMG/PFMG/Jacobi, or MLMG). PCG+PFMG and
# GMRES+PFMG are the textbook-best HYPRE combinations for Poisson-like problems;
# `mlmg` is the AMReX-native matrix-free path that avoids HYPRE entirely.
combos = [
    ("pcg",            "pcg",       None),
    ("pcg+pfmg",       "pcg",       "pfmg"),
    ("pcg+smg",        "pcg",       "smg"),
    ("flexgmres+pfmg", "flexgmres", "pfmg"),
    ("gmres",          "gmres",     None),
    ("gmres+pfmg",     "gmres",     "pfmg"),
    ("bicgstab",       "bicgstab",  None),
    ("smg",            "smg",       None),
    ("pfmg",           "pfmg",      None),
    ("mlmg",           "mlmg",      None),
]

records = []
for label, s, pc in combos:
    kwargs = dict(phase=0, direction="z", solver=s, max_grid_size=32, verbose=0)
    if pc is not None:
        kwargs["preconditioner"] = pc
    try:
        t0 = time.perf_counter()
        res = oi.tortuosity(data_medium, **kwargs)
        dt = time.perf_counter() - t0
        records.append({"label": label, "solver": s, "precond": pc, "t": dt,
                        "iters": res.iterations, "tau": res.tortuosity,
                        "ok": res.solver_converged})
        print(f"  {label:18s}  t={dt:6.2f}s  iters={res.iterations:4d}  tau={res.tortuosity:.4f}")
    except TypeError as e:
        if "preconditioner" in str(e) and pc is not None:
            print(f"  {label:18s}  SKIP — wheel predates preconditioner plumbing")
            records.append({"label": label, "solver": s, "precond": pc,
                            "t": np.nan, "iters": np.nan, "tau": np.nan, "ok": False})
            continue
        raise
    except Exception as e:
        records.append({"label": label, "solver": s, "precond": pc,
                        "t": np.nan, "iters": np.nan, "tau": np.nan, "ok": False})
        print(f"  {label:18s}  FAILED — {e}")

df_solvers = pd.DataFrame(records)
# Back-compat: some downstream code (§10) historically used a "solver" column
# storing the label; expose the label as df_solvers["solver"] alias the
# already-correct atomic name in `solver` (the actual HYPRE/MLMG solver
# string). Rename to keep both.
df_solvers

In [ ]:
ok = df_solvers.dropna(subset=["t"])
if not ok.empty:
    best_row = ok.loc[ok["t"].idxmin()]
    best_label = best_row["label"]
    best_solver = best_row["solver"]
    best_precond = best_row["precond"]
else:
    best_label = "pcg"
    best_solver = "pcg"
    best_precond = None

fig, ax1 = plt.subplots(figsize=(11, max(4, 0.5 * len(df_solvers))))
colors = ["#2ecc71" if c else "#e74c3c" for c in df_solvers["ok"]]
y = np.arange(len(df_solvers))

ax1.barh(y, df_solvers["t"], color=colors, alpha=0.85, edgecolor="white")
ax1.set_yticks(y)
ax1.set_yticklabels(df_solvers["label"])
ax1.set_xlabel("Wall time (s)", color="#2c3e50")
ax1.set_title(f"Solver comparison — 64³, best by wall time: {best_label}",
              fontweight="bold")
ax1.invert_yaxis()

# Overlay iteration counts on a twin axis — this is the key diagnostic.
# A solver with few iterations but expensive each (multigrid) can still
# win overall; wall time alone hides that.
ax2 = ax1.twiny()
ax2.plot(df_solvers["iters"], y, "D", color="#8e44ad", markersize=9,
         markeredgecolor="white", markeredgewidth=1.2, zorder=5)
ax2.set_xlabel("Iterations", color="#8e44ad")
ax2.tick_params(axis="x", colors="#8e44ad")

# Annotate each row with iter count and per-iter cost
for i, (_, row) in enumerate(df_solvers.iterrows()):
    if row["ok"] and not np.isnan(row["t"]):
        per_iter = row["t"] / max(row["iters"], 1)
        ax1.text(row["t"] * 1.02, i,
                 f" {int(row['iters'])} iters  ({per_iter*1000:.0f} ms/iter)",
                 va="center", fontsize=8, color="#2c3e50")

from matplotlib.patches import Patch
legend = [
    Patch(facecolor="#2ecc71", label="Converged"),
    Patch(facecolor="#e74c3c", label="Failed"),
    plt.Line2D([0], [0], marker="D", color="#8e44ad", linestyle="None",
               markersize=8, label="Iterations"),
]
ax1.legend(handles=legend, loc="lower right", framealpha=0.9)
plt.tight_layout()
plt.show()

# Flag solvers with few iterations — likely multigrid/preconditioned.
# If one converged in <10 iterations but is slower per iteration, it may
# still be the better choice at larger problem sizes.
low_iter = ok[ok["iters"] < 10]
if not low_iter.empty:
    print("\nSolvers converging in <10 iterations (likely multigrid):")
    for _, r in low_iter.iterrows():
        print(f"  {r['label']:18s}  {int(r['iters'])} iters  {r['t']:.2f}s")
    print("  These scale O(N) and will overtake PCG at larger grids.")

print(f"\nBest solver at 64³ by wall time: {best_label} "
      f"(solver={best_solver}, preconditioner={best_precond})")


### 9b. Scaling validation — does the multigrid preconditioner restore O(N) compute?

§3b found plain PCG scaling as `t(N) ~ N^1.76` — the dominant source of slow large-grid solves, well above any Python or binding overhead. Theory predicts that pairing PCG with a geometric multigrid preconditioner (PFMG or SMG) drops the iteration count to a near-constant, restoring near-linear `O(N)` scaling overall. AMReX's native MLMG is also `O(N)` by construction.

This cell fits `t = a · N^p` across 32³, 64³, 96³, and 128³ for three representative configurations:

* **`pcg`** — plain conjugate gradient (the baseline, super-linear)
* **`pcg+pfmg`** — PCG with HYPRE's PFMG preconditioner
* **`mlmg`** — AMReX's native geometric multigrid

The test problem is a uniform phase-0 block (always percolates, trivially convergent) so each solve is cheap and the differences come purely from the algorithmic scaling.

A `p` value below ~1.2 indicates near-linear scaling; the plain PCG baseline typically lands around `p ≈ 1.7–1.8`.

In [ ]:
scaling_sizes = [32, 64, 96, 128]
scaling_combos = [
    ("pcg",      {"solver": "pcg"}),
    ("pcg+pfmg", {"solver": "pcg", "preconditioner": "pfmg"}),
    ("mlmg",     {"solver": "mlmg"}),
]

scaling_records = []
for label, kw in scaling_combos:
    for N in scaling_sizes:
        arr = np.zeros((N, N, N), dtype=np.int32)  # uniform phase 0 — always percolates
        try:
            t0 = time.perf_counter()
            res = oi.tortuosity(arr, phase=0, direction="z",
                                max_grid_size=min(32, N), verbose=0, **kw)
            dt = time.perf_counter() - t0
            scaling_records.append({"solver": label, "N": N, "t": dt,
                                    "iters": int(res.iterations),
                                    "ok": bool(res.solver_converged)})
            print(f"  {label:10s}  {N:3d}^3  t={dt:7.3f}s  iters={res.iterations:4d}")
        except TypeError as e:
            # Wheel predates the preconditioner kwarg — skip only those rows.
            msg = str(e)
            if ("preconditioner" in msg) and ("preconditioner" in kw):
                print(f"  {label:10s}  {N:3d}^3  SKIP — wheel predates preconditioner kwarg")
                scaling_records.append({"solver": label, "N": N, "t": np.nan,
                                        "iters": 0, "ok": False})
                continue
            raise
        except Exception as e:
            scaling_records.append({"solver": label, "N": N, "t": np.nan,
                                    "iters": 0, "ok": False})
            print(f"  {label:10s}  {N:3d}^3  FAILED — {e}")

df_scale = pd.DataFrame(scaling_records)
df_scale


In [ ]:
# Fit t = a · N^p in log-log per solver, then overlay the curves with an
# O(N) reference line and the O(N^1.76) baseline from §3b.
fig, ax = plt.subplots(figsize=(9, 5.5))
palette = {"pcg": "#e74c3c", "pcg+pfmg": "#2ecc71", "mlmg": "#8e44ad"}
fit_results = {}
for label, _ in scaling_combos:
    sub = df_scale[(df_scale["solver"] == label) & df_scale["ok"]].dropna(subset=["t"])
    if len(sub) < 2:
        fit_results[label] = np.nan
        continue
    p, loga = np.polyfit(np.log(sub["N"].astype(float)), np.log(sub["t"].astype(float)), 1)
    fit_results[label] = p
    ax.loglog(sub["N"], sub["t"], "o-", color=palette.get(label, "#333333"),
              markersize=8, linewidth=2, label=f"{label}  (p={p:.2f})")

# Reference lines — anchored at the first successful measurement
anchor = df_scale[df_scale["ok"]].dropna(subset=["t"]).sort_values("N")
if not anchor.empty:
    N_anchor = float(anchor.iloc[0]["N"])
    t_anchor = float(anchor.iloc[0]["t"])
    Ns = np.array(scaling_sizes, dtype=float)
    ax.loglog(Ns, t_anchor * (Ns / N_anchor) ** 1.0,
              "k--", alpha=0.4, linewidth=1, label="O(N) reference")
    ax.loglog(Ns, t_anchor * (Ns / N_anchor) ** 1.76,
              "r:", alpha=0.5, linewidth=1, label="O(N^1.76) (§3b baseline)")

ax.set_xlabel("Grid size N  (domain = N^3)")
ax.set_ylabel("Wall time (s)")
ax.set_title("Scaling validation — multigrid vs. plain Krylov",
             fontweight="bold")
ax.legend(loc="upper left", framealpha=0.9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

# Verdict against the near-linear target (p < 1.2)
print("\nScaling exponents vs. near-linear target (p < 1.2):")
for label, p in fit_results.items():
    if np.isnan(p):
        print(f"  {label:10s}  n/a  (insufficient data)")
    elif p < 1.2:
        print(f"  {label:10s}  p = {p:.2f}  near-linear — target met")
    else:
        print(f"  {label:10s}  p = {p:.2f}  super-linear — target NOT met")

## 10. `max_grid_size` tuning

Small boxes → better CPU cache + MPI parallelism. Large boxes → better
GPU kernel saturation. Sweep a few values with the best solver.


In [ ]:
from openimpala import ConvergenceError

grid_sizes = [8, 16, 32, 64]
grid_rows = []
# Re-use the winner from §9 (best_solver / best_precond). On the porespy blob
# the winner is typically flexgmres+pfmg on GPU; on CPU PCG+SMG.
for gs in grid_sizes:
    kwargs = dict(phase=0, direction="z", solver=best_solver,
                  max_grid_size=gs, verbose=0)
    if best_precond is not None:
        kwargs["preconditioner"] = best_precond
    t0 = time.perf_counter()
    try:
        res = oi.tortuosity(data_medium, **kwargs)
        dt = time.perf_counter() - t0
        tau = res.tortuosity
        grid_rows.append({"max_grid_size": gs, "t": dt, "tau": tau, "ok": True})
        print(f"  max_grid_size={gs:3d}  t={dt:.2f}s  tau={tau:.4f}")
    except ConvergenceError as e:
        dt = time.perf_counter() - t0
        grid_rows.append({"max_grid_size": gs, "t": np.nan, "tau": np.nan, "ok": False})
        print(f"  max_grid_size={gs:3d}  FAILED ({e})")

df_grid = pd.DataFrame(grid_rows)
df_grid_ok = df_grid.dropna(subset=["t"])
if df_grid_ok.empty:
    best_gs = 32
    print("No successful grid sweeps — falling back to max_grid_size=32.")
else:
    best_gs = int(df_grid_ok.loc[df_grid_ok["t"].idxmin(), "max_grid_size"])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(df_grid_ok["max_grid_size"], df_grid_ok["t"], "o-", linewidth=2, markersize=9)
ax.set_xlabel("max_grid_size")
ax.set_ylabel("Wall time (s)")
ax.set_title(f"Grid sweep — 64³, solver={best_label}", fontweight="bold")
plt.tight_layout()
plt.show()
print(f"Optimal max_grid_size: {best_gs}")

## 11. Native-binary comparison — optional

If the native `Diffusion` executable is available on `PATH`, this cell runs it on the same data via a `tiff` dump and compares wall time against the equivalent `oi.tortuosity` call. The difference quantifies how much overhead the Python/pybind11 binding layer adds on top of the C++ solver.

This section is informational — a CTest-driven comparison lives at `tests/benchmarks/`. Skip if no native binary is available; §3 – §5 already isolate the binding-side cost.

In [ ]:
native = shutil.which("openimpala")
if not native:
    print("No native openimpala binary on PATH — skipping comparison.")
else:
    import tifffile, tempfile, textwrap, os
    tmp = tempfile.mkdtemp()
    tif_path = os.path.join(tmp, "data.tif")
    tifffile.imwrite(tif_path, data_medium.astype(np.uint8))
    inputs_path = os.path.join(tmp, "inputs")
    with open(inputs_path, "w") as f:
        f.write(textwrap.dedent(f"""
            filename = {tif_path}
            phase_id = 0
            direction = Z
            solver_type = {best_solver}
            box_size = {best_gs}
            hypre.maxiter = 1000
            hypre.eps = 1e-9
            physics.type = diffusion
            verbose = 0
        """).strip())

    t0 = time.perf_counter()
    res = subprocess.run([native, inputs_path], capture_output=True, text=True, timeout=600)
    t_native = time.perf_counter() - t0

    # The native binary exits non-zero on input/parse errors but still returns
    # quickly — don't compare that 0.1s "win" to a real Python solve.
    if res.returncode != 0:
        print("=" * 70)
        print(f"  ⚠  Native binary exited with code {res.returncode} in {t_native:.2f}s")
        print("=" * 70)
        print("  It did NOT actually solve — skipping the comparison.")
        print()
        print("  stderr tail:")
        for line in (res.stderr or "").strip().splitlines()[-10:]:
            print(f"    {line}")
        if res.stdout:
            print("  stdout tail:")
            for line in res.stdout.strip().splitlines()[-5:]:
                print(f"    {line}")
        print("=" * 70)
        print("  Likely cause: the native binary on PATH expects a different")
        print("  inputs-file schema than what this cell writes (field names or")
        print("  types differ between versions). Fix the inputs file to match,")
        print("  or ignore this comparison — §3/§4 already prove the bindings")
        print("  add negligible overhead.")
    else:
        t0 = time.perf_counter()
        oi.tortuosity(data_medium, phase=0, direction="z",
                      solver=best_solver, max_grid_size=best_gs, verbose=0)
        t_py = time.perf_counter() - t0

        print(f"Native binary:    {t_native:.2f}s")
        print(f"Python binding:   {t_py:.2f}s")
        print(f"Binding overhead: {t_py - t_native:+.2f}s "
              f"({100*(t_py-t_native)/max(t_native,1e-9):+.0f}%)")


## 12. Summary & HPC recommendations


In [ ]:
# Headline numbers from the diagnostics above
stage_total = stages["_total"]
stage_top = max((k for k in stages if not k.startswith("_")), key=lambda k: stages[k])
fixed_overhead = a_fit
naive_mean = float(np.mean(naive))
amort_mean = float(np.mean(amort))

print("=" * 64)
print("  DIAGNOSIS")
print("=" * 64)
print(f"  Build backend:                  {backend.upper()}"
      + (f"   (host GPU: {gpu_name})" if gpu_name else ""))
print(f"  Single-call bottleneck:         {stage_top}")
print(f"                                  ({100*stages[stage_top]/stage_total:.1f}% of {stage_total:.2f}s)")
print(f"  Compute scaling:                O(N^{p_fit:.2f})"
      + ("   (super-linear)" if p_fit > 1.2 else ""))
print(f"  Fixed per-call overhead (fit):  {fixed_overhead:.2f}s"
      + ("   (no meaningful fixed cost)" if not overhead_meaningful else ""))
print(f"  Per-call rebuild tax:           {naive_mean - amort_mean:.2f}s "
      f"({100*(naive_mean-amort_mean)/max(naive_mean,1e-9):.0f}% of naive call)")
print(f"  Best solver:                    {best_solver}")
print(f"  Best max_grid_size:             {best_gs}")
print("=" * 64)

# Headline recommendation — the single most impactful change for this run
print("\n  TOP RECOMMENDATION")
print("  " + "-" * 62)
if gpu_name and not is_gpu_build:
    print(f"  A {gpu_name} is available but the installed wheel is CPU-only.")
    print("  Switch to openimpala-cuda for an expected 10–50× speedup on")
    print("  single-phase 3D diffusion at 128³+. Everything below is")
    print("  secondary until this is resolved.")
elif p_fit > 1.2:
    print(f"  Compute scales as O(N^{p_fit:.2f}). Plain Krylov iterations grow")
    print("  with problem size — switch to a multigrid preconditioner")
    print("  (PFMG/SMG via HYPRE, or AMReX MLMG) to restore O(N).")
else:
    print(f"  Solver dominates wall time. Best configuration on this run:")
    print(f"  solver={best_solver}, max_grid_size={best_gs}.")
    print("  Further speedup needs algorithmic changes or larger hardware.")

# Amortization recommendation — only fires when the rebuild tax is meaningful
if naive_mean > 2 * amort_mean:
    print()
    print("  Per-call setup is a significant fraction of wall time. For sweeps,")
    print("  build the VoxelImage once and reuse it:")
    print()
    print("    from openimpala.facade import _numpy_to_voxelimage")
    print("    from openimpala import _core")
    print("    img = _numpy_to_voxelimage(arr, max_grid_size)")
    print("    s   = _core.TortuosityHypre(img, vf, 0, d, st, '.', 0, 1, 0, False)")
    print("    s.value()")
    print()
    print("  …rather than calling oi.tortuosity(numpy_array, ...) per iteration.")

In [ ]:
inputs_template = f"""# Recommended OpenImpala inputs (generated by profiling notebook)
filename = your_image.tif
phase_id = 0
direction = ALL
solver_type = {best_solver}
box_size = {best_gs}
calculation_method = flow_through
hypre.maxiter = 1000
hypre.eps = 1e-9
physics.type = diffusion
verbose = 1
"""
print(inputs_template)


## Cleanup


In [ ]:
try:
    from openimpala._core import amrex_initialized
    if amrex_initialized():
        session.__exit__(None, None, None)
        print("Session closed.")
    else:
        print("Session already finalized.")
except Exception:
    print("Cleanup done.")
